# Model 1: Base CNN (No Regularization)

## Architecture
- **3 Convolutional Layers**: 32 → 64 → 128 filters
- **Max Pooling**: After each conv layer (2x2)
- **2 Fully Connected Layers**: 256 → 10 classes
- **No Dropout, No Batch Normalization**

## Purpose
This is our **baseline model** to establish a performance benchmark. 
All other models will be compared against this one.

## Expected Behavior
- **Overfitting**: Training accuracy will be high, validation accuracy lower
- **Purpose**: Shows why we need regularization

In [1]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import pickle
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from data_loader import AnimalsDataLoader
from models import Model1_BaseCNN
from train import ModelTrainer

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model configuration
MODEL_NAME = 'model_01_base'
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

print(f"\nModel: {MODEL_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")

Using device: cpu

Model: model_01_base
Batch Size: 64
Epochs: 30
Learning Rate: 0.001


In [2]:
# Load data
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

data_loader = AnimalsDataLoader(
    data_dir='../data/animals',
    batch_size=BATCH_SIZE,
    image_size=224,
    use_english_names=True
)

train_loader, val_loader, test_loader = data_loader.load_data(
    val_ratio=0.15,
    test_ratio=0.15
)

print("\n✓ Data loaded successfully!")
print(f"  Training: {len(train_loader.dataset)} images ({len(train_loader)} batches)")
print(f"  Validation: {len(val_loader.dataset)} images ({len(val_loader)} batches)")
print(f"  Test: {len(test_loader.dataset)} images ({len(test_loader)} batches)")


LOADING DATA
✓ Detected Italian class names. Mapping to English:
  cane -> dog
  cavallo -> horse
  elefante -> elephant
  farfalla -> butterfly
  gallina -> chicken
  gatto -> cat
  mucca -> cow
  pecora -> sheep
  ragno -> spider
  scoiattolo -> squirrel

✓ Loaded 26179 images from 10 classes
  Display names: dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirrel

Class Distribution:
  dog: 4863 images
  horse: 2623 images
  elephant: 1446 images
  butterfly: 2112 images
  chicken: 3098 images
  cat: 1668 images
  cow: 1866 images
  sheep: 1820 images
  spider: 4821 images
  squirrel: 1862 images

Data Split (Stratified):
  Training: 18322 images (70.0%)
  Validation: 3922 images (15.0%)
  Test: 3935 images (15.0%)

Class Distribution per Split:

  Training:
    dog: 3404 images
    horse: 1836 images
    elephant: 1012 images
    butterfly: 1478 images
    chicken: 2168 images
    cat: 1167 images
    cow: 1306 images
    sheep: 1274 images
    spider: 3374 ima

In [3]:
# Create model
print("\n" + "="*60)
print("CREATING MODEL")
print("="*60)

model = Model1_BaseCNN(num_classes=NUM_CLASSES)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Show model summary (optional)
print(f"\n{model}")


CREATING MODEL

Model Architecture:
  Total parameters: 25,786,186
  Trainable parameters: 25,786,186

Model1_BaseCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=100352, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


In [4]:
# Initialize trainer
print("\n" + "="*60)
print("INITIALIZING TRAINER")
print("="*60)

trainer = ModelTrainer(
    model=model,
    device=device,
    model_name=MODEL_NAME,
    save_dir='../models_pickles'
)

print(f"✓ Trainer initialized")
print(f"  Save directory: {trainer.save_dir}")


INITIALIZING TRAINER
✓ Trainer initialized
  Save directory: ../models_pickles


In [5]:
# Train model
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.0,
    patience=5
)

print(f"\n✅ Training Complete!")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}%")
print(f"  Best Epoch: {trainer.best_epoch}")


STARTING TRAINING

Training: model_01_base
Device: cpu
Epochs: 30 | LR: 0.001 | Weight Decay: 0.0

Epoch 1/30


  Train Loss: 1.8014 | Train Acc: 38.14%
  Val Loss:   1.5207 | Val Acc:   47.86%
  ✓ Best model saved! (Val Acc: 47.86%)
Epoch 2/30


KeyboardInterrupt: 

In [ ]:
# Plot training history
print("\n" + "="*60)
print("TRAINING VISUALIZATION")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(trainer.history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(trainer.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(x=trainer.best_epoch-1, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({trainer.best_epoch})')
axes[0].set_title(f'Loss During Training - {MODEL_NAME}', fontsize=14)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(trainer.history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(trainer.history['val_acc'], label='Val Acc', linewidth=2)
axes[1].axvline(x=trainer.best_epoch-1, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({trainer.best_epoch})')
axes[1].set_title(f'Accuracy During Training - {MODEL_NAME}', fontsize=14)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
os.makedirs('../outputs', exist_ok=True)
plt.savefig(f'../outputs/{MODEL_NAME}_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved to: outputs/{MODEL_NAME}_history.png")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

test_results = trainer.evaluate(test_loader)

print(f"\nTest Results:")
print(f"  Accuracy: {test_results['accuracy']:.2f}%")
print(f"  Total samples tested: {len(test_results['true_labels'])}")

In [ ]:
# Save all results
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

results = {
    'model_name': MODEL_NAME,
    'model_class': model.__class__.__name__,
    'test_accuracy': test_results['accuracy'],
    'best_val_acc': trainer.best_val_acc,
    'best_epoch': trainer.best_epoch,
    'total_epochs': len(trainer.history['train_loss']),
    'history': trainer.history,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
        'num_classes': NUM_CLASSES,
        'total_params': total_params,
        'trainable_params': trainable_params
    },
    'device': str(device)
}

# Save as JSON
json_path = f'../models_pickles/{MODEL_NAME}_results.json'
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: {json_path}")

# Verify pickle was saved
pickle_path = f'../models_pickles/{MODEL_NAME}.pkl'
if os.path.exists(pickle_path):
    size_mb = os.path.getsize(pickle_path) / (1024 * 1024)
    print(f"✓ Model pickle saved: {pickle_path} ({size_mb:.2f} MB)")
else:
    print("❌ Model pickle not found!")

In [ ]:
# Quick summary
print("\n" + "="*60)
print("SUMMARY - MODEL 1: BASE CNN")
print("="*60)

print(f"\n📊 Performance:")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}% (Epoch {trainer.best_epoch})")
print(f"  Test Accuracy: {test_results['accuracy']:.2f}%")

print(f"\n📈 Training Stats:")
print(f"  Total Epochs Trained: {len(trainer.history['train_loss'])}")
print(f"  Final Train Accuracy: {trainer.history['train_acc'][-1]:.2f}%")
print(f"  Final Val Accuracy: {trainer.history['val_acc'][-1]:.2f}%")

print(f"\n🔍 Overfitting Analysis:")
train_val_gap = trainer.history['train_acc'][-1] - trainer.history['val_acc'][-1]
print(f"  Train-Val Gap: {train_val_gap:.2f}%")

if train_val_gap > 15:
    print(f"  ⚠️ Significant overfitting detected (gap > 15%)")
    print(f"  → Regularization will likely help!")
else:
    print(f"  ✓ Reasonable gap. Model is generalizing well.")

print(f"\n💾 Saved Files:")
print(f"  Model Pickle: models_pickles/{MODEL_NAME}.pkl")
print(f"  Results JSON: models_pickles/{MODEL_NAME}_results.json")
print(f"  Training Plot: outputs/{MODEL_NAME}_history.png")

print(f"\n✅ Model 1 complete! Ready for Model 2.")

## Observations (Fill in after training)

### What I Notice
1. **Training Accuracy**: ___%
2. **Validation Accuracy**: ___%
3. **Test Accuracy**: ___%
4. **Overfitting Gap**: ___%

### Key Takeaways
- [ ] The model is overfitting/underfitting
- [ ] Training and validation curves indicate...
- [ ] This baseline will be compared against...

### Next Steps
- Move to Model 2: CNN with Dropout 0.25